# 01 — Transfermarkt: plantilla actual Rayados

**Patrulla pequena**. Antes de descargar las 4 ligas enteras, validamos primero:

1. Que Transfermarkt no banea nuestra IP
2. Que los headers funcionan (devuelve 200)
3. Que el parser extrae los datos correctos

Si esto funciona, escalamos al notebook 02 (4 ligas).

**Tiempo estimado**: 1-2 minutos.

In [1]:
# Celda 1 — Setup
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_fetcher as df_mod
import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print(f'Project root: {PROJECT_ROOT}')
print(f'Cache dir:    {df_mod.DATA_RAW}')

Project root: C:\Users\msanr\Downloads\rayados_scouting_lab_v3_bloque1\rayados_scouting_lab
Cache dir:    C:\Users\msanr\Downloads\rayados_scouting_lab_v3_bloque1\rayados_scouting_lab\data\raw


## Paso 1: Probar conexion basica

Antes de parsear nada, solo verificamos que podemos pedir la pagina sin que TM nos bloquee.

In [3]:
# Celda 2 — Test de conexion
import requests

url = f'{df_mod.TM_BASE}/cf-monterrey/startseite/verein/{df_mod.CLUB_RAYADOS_TM_ID}'
print(f'URL: {url}')
print(f'Pidiendo...')

r = requests.get(url, headers=df_mod.HEADERS_BROWSER, timeout=30)
print(f'\nStatus code: {r.status_code}')
print(f'Content-Type: {r.headers.get("Content-Type")}')
print(f'HTML size: {len(r.text):,} bytes')

if r.status_code == 200:
    print('\n[OK] Transfermarkt responde correctamente.')
    # mini-preview del HTML
    print('\nPrimeros 500 caracteres:')
    print(r.text[:500])
elif r.status_code == 403:
    print('\n[ERROR] 403 Forbidden - TM bloqueo la peticion.')
    print('Posibles causas:')
    print('  - VPN/Proxy activo')
    print('  - IP de tu region bloqueada')
    print('  - Headers insuficientes')
else:
    print(f'\n[ERROR] Status inesperado: {r.status_code}')

URL: https://www.transfermarkt.es/cf-monterrey/startseite/verein/2407
Pidiendo...

Status code: 200
Content-Type: text/html; charset=UTF-8
HTML size: 186,142 bytes

[OK] Transfermarkt responde correctamente.

Primeros 500 caracteres:
<!DOCTYPE html>
<html lang="es">

<head>
    
<script type="text/javascript" data-description="sourcepoint stub code">
    !function () { var e = function () { var e, t = "__tcfapiLocator", a = [], n = window; for (; n;) { try { if (n.frames[t]) { e = n; break } } catch (e) { } if (n === window.top) break; n = n.parent } e || (!function e() { var a = n.document, r = !!n.frames[t]; if (!r) if (a.body) { var i = a.createElement("iframe"); i.style.cssText = "display:none", i.name = t, a.body.append


## Paso 2: Intentar parsear

Si el paso 1 dio 200, probamos el parser completo.

In [2]:
# Celda 3 — Descargar y parsear plantilla
from src.data_fetcher import fetch_tm_club_squad

try:
    squad = fetch_tm_club_squad(force_refresh=True)
    print(f'\n[OK] {len(squad)} jugadores extraidos')
    print('\nPrimeras filas:')
    print(squad.head(10).to_string(index=False))
except RuntimeError as e:
    print(f'[ERROR PARSING] {e}')
    print('\nGuardando HTML para que Claude lo revise...')
    import requests
    url = f'{df_mod.TM_BASE}/cf-monterrey/kader/verein/{df_mod.CLUB_RAYADOS_TM_ID}/saison_id/2025/plus/1'
    r = requests.get(url, headers=df_mod.HEADERS_BROWSER, timeout=30)
    df_mod.save_debug_html(r.text, 'debug_tm_rayados.html')
    print('Adjunta a Claude el archivo data/raw/debug_tm_rayados.html')

  [tm] GET https://www.transfermarkt.es/cf-monterrey/kader/verein/2407/saison_id/2025/plus/1
      status: 200
  [saved] tm_squad__2407.csv  (26 jugadores)

[OK] 26 jugadores extraidos

Primeras filas:
        jugador  tm_player_id dorsal       posicion_tm fecha_nacimiento  edad nacionalidad  valor_mercado_meur
  Santiago Mele        320739     25           Portero       06/09/1997    28      Uruguay                2.20
  Luis Cárdenas        280379     22           Portero       15/09/1993    32       México                0.25
  Víctor Guzmán        686444      4   Defensa central       07/03/2002    24       México                5.00
 Carlos Salcedo        256866     13   Defensa central       29/09/1993    32       México                1.00
   Luis Sánchez        670185     23   Defensa central       03/05/2000    26       México                1.00
   César Bustos        944086     34   Defensa central       27/08/2005    20       México                0.10
Gerardo Arteaga      

## Paso 3: Inspeccion de calidad

Si el paso 2 funciono, verificar que los datos tienen sentido.

In [ ]:
# Celda 4 — Sanity checks
if 'squad' in dir() and squad is not None and not squad.empty:
    print('Resumen de la plantilla:')
    print(f'  Total jugadores: {len(squad)}')
    print(f'  Edad media: {squad["edad"].mean():.1f}')
    print(f'  Edad min/max: {squad["edad"].min()} / {squad["edad"].max()}')
    print(f'  Valor total plantilla: {squad["valor_mercado_meur"].sum():.1f} ME')
    print(f'  Valor maximo individual: {squad["valor_mercado_meur"].max():.1f} ME')
    print(f'  Nacionalidades: {squad["nacionalidad"].value_counts().to_dict()}')
    print(f'  Posiciones detectadas: {squad["posicion_tm"].value_counts().to_dict()}')
    
    print('\nJugador mas valioso:')
    top = squad.nlargest(5, 'valor_mercado_meur')[['jugador','edad','posicion_tm','valor_mercado_meur']]
    print(top.to_string(index=False))
    
    print('\n[OK] Si los valores son plausibles, el scraping funciona.')
    print('Pasos siguientes: notebook 02 para escalar a las 4 ligas.')